In [ ]:
LAWYER_EVAL_PROMPT = """你是一个专业的法律监督AI。你的任务是评估模拟法庭案件中律师的表现。
你必须将模拟的庭审记录与真实的案件档案（Ground Truth）进行对比。

需要评估的目标律师: {lawyer_role}

指示：
1. 识别 {lawyer_role} 在庭审记录中提出的所有法律论点、引用的法律条文和事实要点。
2. 将这些与提供的“真实案件档案”进行比对。指出 {lawyer_role} 成功提出的关键要点/法律条文，以及他们完全遗漏的关键要点/法律条文。
3. 批判性地分析 {lawyer_role} 的辩论能力（例如：逻辑连贯性、说服力、应对对方律师的能力），并给出一个 0 到 10 的评分。

输入数据：
--- 真实案件档案 (GROUND TRUTH) ---
{ground_truth}

--- 庭审记录 (TRANSCRIPT) ---
{transcript}

输出格式：
你必须只返回一个有效的JSON对象，包含以下确切的键：
{{
  "points_cited": "律师成功辩护的正确事实和法律条文的简明列表。",
  "points_missed": "真实案件档案中律师未能提及的关键事实或法律条文的简明列表。",
  "argumentation_and_reasons": "对其辩论能力的简短评价，说明其逻辑的强弱原因。",
  "argumentation_score": 0到10之间的整数评分 (例如 8)
}}"""

JUDGE_EVAL_PROMPT = """
你是一个专业的司法审查AI。你的任务是评估模拟法官的表现。
你必须将庭审记录中法官的最终判决和说理，与真实的案件档案（Ground Truth）进行对比。

指示：
1. 提取模拟法官的最终判决结果及法官给出的理由。
2. 从“真实案件档案”中提取真实的判决结果和真实理由。
3. 识别模拟法官在裁决中遗漏的任何重大法律理由或事实。
4. 评估“律师意见采纳度”：模拟法官是否适当认可并权衡了原告和被告在庭审中提出的有效观点，还是忽视了他们？
5. 给出一个 0 到 10 的整体判决准确度评分。

输入数据：
--- 真实案件档案 (GROUND TRUTH) ---
{ground_truth}

--- 庭审记录 (TRANSCRIPT) ---
{transcript}

输出格式：
你必须只返回一个有效的JSON对象，包含以下确切的键：
{{
  "judgement_sentence": "模拟法官给出的确切最终判决。",
  "given_reasons": "模拟法官声称做出此裁决的理由摘要。",
  "real_reasons": "真实案件档案中的实际判决和理由。",
  "missing_reasons": "模拟法官未能考虑的决定真实判决的关键法律或事实。",
  "judge_lawyer_reception": "关于法官在模拟过程中如何吸收和裁决律师提出的具体论点的简短评价。",
  "judgement_score": 0到10之间的整数评分 (例如 9)
}}
"""

CASE_SUMMARY_PROMPT = """你是一个法律摘要AI。

指示：
阅读以下“真实案件档案”，并将核心冲突浓缩为一句话（最多不超过15个字）。
只关注核心指控或纠纷。不要包含判决结果。

输入数据：
--- 真实案件档案 (GROUND TRUTH) ---
{ground_truth}

输出格式：
只返回摘要的纯文本字符串，不要有任何格式或JSON。
例如："关于未交付500根钢管的合同纠纷。"""

In [ ]:
from openai import OpenAI
import json

key = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=key,
    # OpenRouter highly recommends providing these headers to route requests properly
    default_headers={
        "HTTP-Referer": "http://localhost:8000", # Change to your actual site/app URL
        "X-Title": "CourtSimulationTool",        # Change to your app name
    }
)

class LegalAgent:
    def __init__(self, role, base_prompt):
        self.role = role
        self.base_prompt = base_prompt
        self.memory = "" 

    def generate_response(self, current_case_details, transcript):
        prompt = f"案件详情：{current_case_details}\n庭审记录：\n{transcript}\n\n现在轮到你作为【{self.role}】发言："
        
        system_content = self.base_prompt
        if self.memory:
            system_content += f"\n\n--- 你的过往经验 (记忆) ---\n{self.memory}\n"

        response = client.chat.completions.create(
            model="qwen/qwen3-4b", # Uses Qwen 3 4B
            messages=[
                {"role": "system", "content": system_content},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        return response.choices[0].message.content


class LogicEvaluator:
    def evaluate_lawyer(self, transcript, ground_truth, lawyer_role):
        # Format the Chinese prompt provided above
        eval_prompt = LAWYER_EVAL_PROMPT.format(
            lawyer_role=lawyer_role, 
            ground_truth=ground_truth, 
            transcript=transcript
        )
        
        response = client.chat.completions.create(
            model="google/gemini-3.1-flash-lite", # Uses Gemini 3.0 Flash
            messages=[
                {"role": "user", "content": eval_prompt}
            ],
            # OpenRouter supports forcing JSON object format for modern models 
            response_format={"type": "json_object"}, 
            temperature=0.1 # Keep temperature low for evaluations
        )
        
        result_text = response.choices[0].message.content
        return json.loads(result_text)

    def evaluate_judge(self, transcript, ground_truth, lawyer_role):
        # Format the Chinese prompt provided above
        eval_prompt = JUDGE_EVAL_PROMPT.format(
            lawyer_role=lawyer_role, 
            ground_truth=ground_truth, 
            transcript=transcript
        )
        
        response = client.chat.completions.create(
            model="google/gemini-3.1-flash-lite", # Uses Gemini 3.0 Flash
            messages=[
                {"role": "user", "content": eval_prompt}
            ],
            # OpenRouter supports forcing JSON object format for modern models 
            response_format={"type": "json_object"}, 
            temperature=0.1 # Keep temperature low for evaluations
        )
        
        result_text = response.choices[0].message.content
        return json.loads(result_text)

    def generate_case_summary(self, ground_truth):
        prompt = CASE_SUMMARY_PROMPT.format(ground_truth=ground_truth)
        response = client.chat.completions.create(
            model="google/gemini-3.1-flash-lite", 
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        return response.choices[0].message.content.strip()

class CourtSimulation:
    def __init__(self, plaintiff, defendant, judge, max_rounds=3):
        self.plaintiff = plaintiff
        self.defendant = defendant
        self.judge = judge
        self.max_rounds = max_rounds

    def run_case(self, case_details):
        transcript = ""
        verdict_reached = False

        for round_num in range(self.max_rounds):
            print(f"--- Round {round_num + 1} ---")
            
            # Plaintiff Turn
            p_resp = self.plaintiff.generate_response(case_details, transcript)
            transcript += f"Plaintiff: {p_resp}\n\n"

            # Defendant Turn
            d_resp = self.defendant.generate_response(case_details, transcript)
            transcript += f"Defendant: {d_resp}\n\n"

            # Judge Turn
            j_resp = self.judge.generate_response(case_details, transcript)
            transcript += f"Judge: {j_resp}\n\n"

            # Check if judge ended it early (e.g., keyword detection or LLM structured output)
            if "<FINAL_JUDGMENT>" in j_resp or round_num == self.max_rounds - 1:
                if "<FINAL_JUDGMENT>" not in j_resp:
                    # Force judge to conclude if max rounds reached
                    forced_prompt = transcript + "\nSystem: Maximum rounds reached. Judge, please formulate your FINAL_JUDGMENT now."
                    j_resp = self.judge.generate_response(case_details, forced_prompt)
                    transcript += f"Judge (Forced Final): {j_resp}\n\n"
                
                verdict_reached = True
                break
                
        return transcript
    



In [ ]:
from openai import OpenAI
import os



